# Week 1: call the model from code

This week you add one thing. Your own Python code sends text to the model and
reads the answer back.

That is the whole job. Nothing is sorted into a fixed shape yet. Nothing is
checked yet. Those come in later weeks.

An agent is a loop around this one call. It is not magic. So the call comes
first, and it comes on its own.

## Your work is saved

Two promises. Both hold all term.

Your work in this notebook server is saved for the whole course. Nothing you
write here disappears when you close the tab.

Every later week opens with a starter notebook. That notebook already holds a
working version of everything through the week before. You carry your own code
forward where it works. Where it does not, the starter version is right there
and runs.

So a hard week costs you that week's points. It does not end the course.

Run the cells in order, from the top.


## How this notebook talks to the model

The model runs on another computer. Your notebook sends it a request. The
model sends back a reply.

A **request** is a dictionary. It holds the model name, your text, and a few
settings. A reply is a dictionary too. Your answer sits inside it.

Three words you will use all term.

- A **prompt** is the text you send.
- A **token** is a chunk of text, about four characters long. The model counts
  its work in tokens.
- **Temperature** is a setting on the call. A low one usually keeps answers
  steadier. A high one often lets them vary more, but a small run may show a
  different pattern.

Your API key is placed in this notebook for you. You never type it, and no cell
in this course prints it.


In [ ]:
# Configuration. Pre-written. Run this cell first, every time you open the
# notebook. One line in it is yours to fill in: your key, just below.

# This notebook runs Python inside your browser. `requests` is not there until
# it is asked for. The next line fetches it from the course server. It takes a
# few seconds the first time you run it in a session and nothing after that.
try:
    import piplite
    await piplite.install("requests")
except ImportError:
    pass  # a normal Jupyter server already has requests

import json
import os
import sys
from pathlib import Path

import requests

# =====================================================================
#  PASTE YOUR KEY BETWEEN THE QUOTES ON THE NEXT LINE, THEN RUN THIS CELL
# =====================================================================
API_KEY = ""
# =====================================================================
# The course's own checks run these notebooks without a person present, and
# supply the key through the environment instead. A browser has no environment,
# so this line changes nothing for a student.
if not API_KEY:
    API_KEY = os.environ.get("ITSY1377_API_KEY", "")
# The key is yours alone and it carries your own spending limit. Do not put it
# in a discussion post, an email, a screenshot, or anyone else's notebook.

# Which notebook this is. It rides along with every request, so the course can
# see which week the work came from. This is the only line in this cell that
# differs between the seven lab notebooks.
NOTEBOOK = "w1-lab"

# Where the model lives, and which model answers. Both are set for you, and
# you do not change them. The course's own automated checks point these at a
# test server instead; a browser has no environment, so nothing changes here.
#
# This week names a larger model than the rest of the course does. The reason
# is the temperature comparison further down: the smaller model answers this
# week's short prompt the same way at every setting, so there would be nothing
# to see. From week 2 on you use the smaller one, which is what a real queue
# would run. A model is a setting, not a fixture, and the right one depends on
# what you are asking it to show you.
BASE_URL = os.environ.get("ITSY1377_BASE_URL", "https://api.bashtion.dev/v1")
MODEL = os.environ.get("ITSY1377_MODEL", "granite-8b")

# Extra fields added to every request. Some models think before they answer,
# which is slow and costs tokens you never see. This field switches that off.
MODEL_EXTRAS = {"reasoning_effort": "none"}

# Where the course files sit. This looks upward from the folder the notebook
# is open in until it finds them.
COURSE_DIR = Path.cwd().resolve()
for _place in [Path.cwd().resolve()] + list(Path.cwd().resolve().parents):
    if (Path(_place) / "corpus" / "emails" / "teaching").is_dir():
        COURSE_DIR = Path(_place)
        break

EMAIL_DIR = COURSE_DIR / "corpus" / "emails" / "teaching"
DATA_DIR = COURSE_DIR / "corpus" / "data"
# Two folders hold code the notebooks import: the email parser sits with the
# corpus, and the triage tools sit with the lab. Both go on the path, so a
# later week that imports the tools finds them without editing this cell.
TOOLS_DIR = COURSE_DIR / "corpus" / "tools"
LAB_TOOLS_DIR = COURSE_DIR / "lab" / "tools"
for _dir in (TOOLS_DIR, LAB_TOOLS_DIR):
    if _dir.is_dir() and str(_dir) not in sys.path:
        sys.path.insert(0, str(_dir))

print("Notebook:   ", NOTEBOOK)
print("Model:      ", MODEL)
print("Endpoint:   ", BASE_URL)
print("Extra fields:", MODEL_EXTRAS)
print("API key:    ", "pasted in" if API_KEY else "NOT PASTED IN YET")
print("Email files:", "found" if EMAIL_DIR.is_dir() else "NOT FOUND", EMAIL_DIR)

if not API_KEY:
    print()
    print("You have not pasted your key in yet.")
    print('Scroll up in this cell to the line that reads   API_KEY = ""')
    print("Paste your key between the two quotation marks, then run this cell again.")


In [ ]:
# The token counter. Pre-written. You do not edit this cell.
#
# Every reply carries a usage block with three numbers: tokens in the prompt,
# tokens in the answer, and the total. This cell reads those reported numbers.
# It never measures the length of the answer text. Some models spend tokens
# thinking and never show that text, so measuring the text under-reports.

USAGE = {"prompt": 0, "answer": 0, "total": 0, "calls": 0}
RUN_USAGE = {"prompt": 0, "answer": 0, "total": 0, "calls": 0}
LAST_CALL = {"stop_reason": None, "reasoning": "", "reply": None,
             "answer_was_empty": None}

# Set this to False to hide the one-line report after each call. The tokens
# are still counted either way.
SHOW_CALL_TOKENS = True


def record_usage(response):
    """Add one reply's reported token numbers into both counters."""
    usage = response.get("usage")
    if not usage:
        if SHOW_CALL_TOKENS:
            print("   token count not reported")
        return
    prompt = usage.get("prompt_tokens", 0)
    answer = usage.get("completion_tokens", 0)
    total = usage.get("total_tokens", prompt + answer)
    for counter in (USAGE, RUN_USAGE):
        counter["prompt"] = counter["prompt"] + prompt
        counter["answer"] = counter["answer"] + answer
        counter["total"] = counter["total"] + total
        counter["calls"] = counter["calls"] + 1
    if SHOW_CALL_TOKENS:
        print(f"   this call: {total} tokens. Session total so far: {USAGE['total']}.")


def show_tokens():
    """Print the running totals for this notebook session."""
    print(f"Tokens this session: {USAGE['total']} total, "
          f"{USAGE['prompt']} sent, {USAGE['answer']} answered, "
          f"over {USAGE['calls']} calls.")


def reset_tokens():
    """Set the session counter back to zero."""
    for key in USAGE:
        USAGE[key] = 0


def start_run():
    """Set the per-email counter back to zero."""
    for key in RUN_USAGE:
        RUN_USAGE[key] = 0


show_tokens()

In [ ]:
# Pre-written. You do not edit this cell.
#
# send_request() is the only place in this course that talks to the network.
# save_artifact() writes your submission file.

import time

REQUEST_TIMEOUT = 120  # seconds to wait for one answer

NO_REPLY_TEXT = ("No answer came back. The request did not reach the model, or "
                 "the model refused it. Read the lines printed above this one.")

# The full list of things to check prints once per session. After that a
# failed call prints one line, so six failed calls do not fill the screen.
TROUBLE_SHOWN = False


def send_request(payload):
    """Send one request to the model. Hand back the whole reply as a dictionary."""
    global TROUBLE_SHOWN
    # Tell the proxy which notebook this came from. The course reads this to
    # see what each week actually costs. It changes nothing about the answer.
    payload = dict(payload)
    payload.setdefault("metadata", {})["tags"] = [NOTEBOOK]

    url = BASE_URL.rstrip("/") + "/chat/completions"
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = "Bearer " + API_KEY

    reply = None
    trouble = ""
    try:
        response = requests.post(url, headers=headers, json=payload,
                                 timeout=REQUEST_TIMEOUT)
        # 429 means you sent requests faster than your key is allowed to. It is
        # not a broken key and not a broken notebook. The only fix is to wait,
        # so this waits for you rather than printing advice you cannot use.
        attempt = 0
        while response.status_code == 429 and attempt < 3:
            attempt = attempt + 1
            wait = 20 * attempt
            print(f"   Too many requests too quickly. Waiting {wait} seconds,"
                  f" then trying again ({attempt} of 3).")
            print("   Nothing is broken. Your key has a limit on how fast it")
            print("   may send, and a busy cell can reach it.")
            time.sleep(wait)
            response = requests.post(url, headers=headers, json=payload,
                                     timeout=REQUEST_TIMEOUT)
        if response.status_code == 200:
            reply = response.json()
        elif response.status_code == 429:
            trouble = ("Your key sent more than it is allowed to in one minute,"
                       " three times over. Wait two minutes, then run this cell"
                       " again. Nothing is broken and nothing is lost.")
        elif response.status_code == 401:
            trouble = ("The model service did not accept your key. Check the"
                       " configuration cell: the key goes between the two"
                       " quotation marks, with no spaces around it.")
        else:
            trouble = f"The model service answered with status {response.status_code}."
            try:
                trouble = trouble + " It said: " + response.json()["error"]["message"]
            except Exception:
                trouble = trouble + " It said: " + response.text[:200]
    except requests.exceptions.Timeout:
        trouble = f"The model took longer than {REQUEST_TIMEOUT} seconds, so the notebook stopped waiting."
    except requests.exceptions.ConnectionError:
        trouble = "The model service did not answer. It may be switched off."
    except Exception as problem:
        trouble = "The request did not go through. " + str(problem)[:150]

    if reply is None:
        print("   This call did not work. " + trouble)
        if not TROUBLE_SHOWN:
            print("   Check these three things, in order:")
            print("   1. Check the configuration cell. Your key goes between the quotes.")
            print("   2. Ask whether the model service is up. Your instructor will know.")
            print("   3. Check that your browser still has a connection.")
            TROUBLE_SHOWN = True
        LAST_CALL["stop_reason"] = "no_reply"
        LAST_CALL["reasoning"] = ""
        LAST_CALL["reply"] = {"error": trouble}
        LAST_CALL["answer_was_empty"] = True
        return {"choices": [{"message": {"role": "assistant",
                                         "content": NO_REPLY_TEXT},
                             "finish_reason": "no_reply"}],
                "usage": None,
                "error": trouble}

    # Make sure the reply has the shape the rest of the notebook expects.
    if not reply.get("choices"):
        reply["choices"] = [{"message": {"role": "assistant", "content": ""},
                             "finish_reason": "no_choices"}]
    choice = reply["choices"][0]
    if not isinstance(choice.get("message"), dict):
        choice["message"] = {"role": "assistant", "content": ""}
    message = choice["message"]

    # Some models write their thinking before the answer. It can arrive in a
    # field of its own, or wrapped in tags inside the answer text. Either way
    # the thinking is moved into LAST_CALL and only the answer is handed back.
    content = message.get("content") or ""
    thinking = message.get("reasoning") or message.get("reasoning_content") or ""
    if "</think>" in content:
        head, _, tail = content.partition("</think>")
        thinking = thinking + head.replace("<think>", "")
        content = tail
    content = content.strip()

    # Did the model write any answer text of its own? Read that here, before
    # the lines below put a notice in an empty slot. Once the notice is in,
    # nothing can tell an empty answer from the notice standing in for one.
    answer_was_empty = not content

    finish = choice.get("finish_reason")
    if not content and finish == "length":
        content = ("No answer text came back. The model used all of its room "
                   "before it answered. Ask for a shorter answer, or raise "
                   "max_tokens.")
        print("   The model ran out of room before it answered.")
    elif not content:
        content = ("No answer text came back. The reply held no answer. The "
                   "reason the model gave was: " + str(finish))
        print("   The model sent no answer text. Reason: " + str(finish))

    message["content"] = content
    LAST_CALL["stop_reason"] = finish
    LAST_CALL["reasoning"] = thinking
    LAST_CALL["reply"] = reply
    LAST_CALL["answer_was_empty"] = answer_was_empty
    record_usage(reply)
    return reply


def save_artifact(filename, content):
    """Write the submission file. Print its full path and its first lines."""
    if isinstance(content, str):
        text = content
    else:
        text = json.dumps(content, indent=2)
    path = Path.cwd().resolve() / filename
    path.write_text(text, encoding="utf-8")
    print("Saved this file:")
    print("   " + str(path))
    print("To hand it in: find it in the file list on the left of this page,")
    print("right-click it, choose Download, then upload it to Canvas.")
    print("First lines of it:")
    for line in text.splitlines()[:8]:
        print("   " + line)
    return str(path)


def budget_left(quiet=False):
    """Say how much of your allowance is left. This call is free.

    It does not reach the model, so it spends nothing and does not count
    against your requests-per-minute limit. Both were measured.
    """
    if not API_KEY:
        if not quiet:
            print("Budget: no key pasted in yet, so there is nothing to read.")
        return None
    try:
        reply = requests.get(BASE_URL.rstrip("/") + "/key/info",
                             headers={"Authorization": "Bearer " + API_KEY},
                             timeout=30)
        info = reply.json()["info"]
    except Exception as problem:
        if not quiet:
            print("Budget: could not be read. " + str(problem)[:130])
        return None
    left = info["max_budget"] - info["spend"]
    if not quiet:
        print(f"Budget: ${info['max_budget']:.2f} allowed, "
              f"${info['spend']:.6f} spent, ${left:.6f} left.")
        print("   Spending catches up about ten seconds behind, so the answer")
        print("   you just received may not be counted in that yet.")
    return left


def backup_my_work(filename=None):
    """Bundle your notebooks and submission files into one file to download.

    Your work lives only in this browser. There is no copy on the server.
    Clearing site data, using a different computer, or a lab machine that
    resets itself will lose it. Run this at the end of every week, download
    the file it names, and keep it somewhere you trust.

    Save the notebook first, with Ctrl+S. This bundles what is on disk, not
    what is on screen.
    """
    import datetime
    import zipfile
    if sys.platform != "emscripten":
        print("Backup: not needed here. This is a real Jupyter server with real files.")
        return None
    here = Path.cwd().resolve()
    if filename is None:
        filename = "itsy1377-backup-" + datetime.date.today().isoformat() + ".zip"
    KEEP = (".ipynb", ".txt", ".json", ".md", ".csv")
    added = []
    with zipfile.ZipFile(here / filename, "w", zipfile.ZIP_DEFLATED) as bundle:
        for item in sorted(here.rglob("*")):
            if not item.is_file():
                continue
            rel = item.relative_to(here)
            # corpus and lab are shipped with the course and come back on their
            # own. Only your own work goes in the bundle.
            if rel.parts and rel.parts[0] in ("corpus", "lab"):
                continue
            if item.suffix.lower() not in KEEP:
                continue
            bundle.write(item, str(rel))
            added.append(str(rel))
    print(f"Backup: {len(added)} file(s) bundled into {filename}")
    for name in added[:10]:
        print("   " + name)
    if len(added) > 10:
        print(f"   and {len(added) - 10} more")
    print("   Find it in the file list on the left, right-click it, choose Download.")
    print("   Keep it somewhere that is not this browser.")
    return filename


def restore_my_work(filename):
    """Put a downloaded backup back. Drag it into the file list first.

    Anything already here with the same name is kept, renamed, rather than
    written over.
    """
    import zipfile
    here = Path.cwd().resolve()
    bundle_path = here / filename
    if not bundle_path.is_file():
        print("Restore: there is no file called " + filename + " here.")
        print("   Drag your backup into the file list on the left, then run this again.")
        return None
    notes = []
    with zipfile.ZipFile(bundle_path) as bundle:
        for name in bundle.namelist():
            out = here / name
            out.parent.mkdir(parents=True, exist_ok=True)
            if out.exists():
                kept = out.with_name(out.stem + "-before-restore" + out.suffix)
                out.replace(kept)
                notes.append(f"{name}   (what was here is kept as {kept.name})")
            else:
                notes.append(name)
            out.write_bytes(bundle.read(name))
    print(f"Restore: {len(notes)} file(s) put back:")
    for line in notes:
        print("   " + line)
    print("   Close and reopen any notebook you restored, so the page reads it again.")
    return notes


print("send_request(), save_artifact(), budget_left() and backup_my_work() are ready.")
budget_left()

## What `ask_model()` has to do

Now you write the one function this week adds. Its name is `ask_model()`.

It takes three things.

- `prompt`, the text you want to send.
- `max_tokens`, the most tokens the answer may use. The default is 512.
- `temperature`, how much the answer may vary. The default is 0.2.

It hands back one thing. That is the answer text, as a string.

Inside, it does four small jobs.

1. Build one outer request dictionary named `body`. It needs four keys:
   `model`, `messages`, `max_tokens` and `temperature`. Use the `MODEL` from
   the configuration cell.
   `messages` is a list holding one dictionary, and that dictionary has a
   `role` of `"user"` and a `content`.
2. Merge `MODEL_EXTRAS` into the outer `body` dictionary. The `.update()`
   method does it in one line. Do not put the extras inside the message.
3. Call `send_request()` with `body`. Keep what it hands back.
4. Read the answer text out of the reply, and return it. The text sits at
   `reply["choices"][0]["message"]["content"]`.

Eight to ten lines. Sidebar 1C builds a dictionary that holds a list and
merges a second dictionary into the outer one, which is the shape you need
here. Week 0 sections 2 and 3 cover calls and dictionaries.


In [ ]:
# SOLUTION. Not in the student notebook.
#
# ask_model() sends one prompt and hands back the answer text.

def ask_model(prompt, max_tokens=512, temperature=0.2):
    body = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    body.update(MODEL_EXTRAS)
    reply = send_request(body)
    return reply["choices"][0]["message"]["content"]


In [ ]:
# Pre-written. A check on the function you just wrote.
#
# First, a fake sender captures the request without contacting the model.
# The marker values prove that TODO 1 forwards what it is handed instead of
# using hard-coded settings. The real sender and extras are always restored.

CHECK_PAYLOAD = {}
CHECK_MAX_TOKENS = "max_tokens check marker"
CHECK_TEMPERATURE = "temperature check marker"
CHECK_REASONING = "reasoning_effort check marker"


def capture_request(payload):
    CHECK_PAYLOAD.update(payload)
    return {"choices": [{"message": {"content": "request captured"}}]}


REAL_SEND_REQUEST = send_request
REAL_MODEL_EXTRAS = MODEL_EXTRAS
CHECK_ANSWER = None
CHECK_PROBLEM = ""
try:
    send_request = capture_request
    MODEL_EXTRAS = {"reasoning_effort": CHECK_REASONING}
    CHECK_ANSWER = ask_model("request check",
                             max_tokens=CHECK_MAX_TOKENS,
                             temperature=CHECK_TEMPERATURE)
except Exception as problem:
    CHECK_PROBLEM = str(problem)
finally:
    send_request = REAL_SEND_REQUEST
    MODEL_EXTRAS = REAL_MODEL_EXTRAS

CHECK_REQUEST_OK = True
CHECK_MESSAGES = CHECK_PAYLOAD.get("messages", [])
CHECK_MESSAGE = {}
if isinstance(CHECK_MESSAGES, list) and len(CHECK_MESSAGES) > 0:
    if isinstance(CHECK_MESSAGES[0], dict):
        CHECK_MESSAGE = CHECK_MESSAGES[0]

if CHECK_PROBLEM:
    CHECK_REQUEST_OK = False
    print("TODO 1 raised an error while building the request:", CHECK_PROBLEM)
if CHECK_ANSWER is None:
    CHECK_REQUEST_OK = False
    print("TODO 1 returned None. Add the return line in your ask_model() cell,")
    print("run TODO 1, then run this check again. Week 0 cell 53,")
    print("'The difference between return and print,' shows this exact trap.")
elif not isinstance(CHECK_ANSWER, str):
    CHECK_REQUEST_OK = False
    print("TODO 1 must return the answer text as a string.")
elif CHECK_ANSWER.startswith("ask_model() is not written"):
    CHECK_REQUEST_OK = False
    print("Write ask_model() in TODO 1, then run that cell and this one again.")
if CHECK_PAYLOAD.get("max_tokens") != CHECK_MAX_TOKENS:
    CHECK_REQUEST_OK = False
    print("TODO 1 did not pass max_tokens into the outgoing request.")
if CHECK_PAYLOAD.get("temperature") != CHECK_TEMPERATURE:
    CHECK_REQUEST_OK = False
    print("TODO 1 did not pass temperature into the outgoing request.")
if CHECK_PAYLOAD.get("reasoning_effort") != CHECK_REASONING:
    CHECK_REQUEST_OK = False
    print("TODO 1 did not merge MODEL_EXTRAS into the outer request.")
if CHECK_PAYLOAD.get("model") != MODEL:
    CHECK_REQUEST_OK = False
    print("TODO 1 did not put MODEL in the outgoing request.")
if CHECK_MESSAGE.get("role") != "user":
    CHECK_REQUEST_OK = False
    print("TODO 1 did not build the user message in the expected place.")
if CHECK_MESSAGE.get("content") != "request check":
    CHECK_REQUEST_OK = False
    print("TODO 1 did not put the prompt into the user message.")
if "reasoning_effort" in CHECK_MESSAGE:
    CHECK_REQUEST_OK = False
    print("Move MODEL_EXTRAS out of the message and into the outer request.")

if CHECK_REQUEST_OK:
    print("Request check passed: the settings reached the outgoing payload.")
    CHECK_QUESTION = "Name three primary colors. Answer in one short sentence."
    print("Question:", CHECK_QUESTION)
    CHECK_ANSWER = ask_model(CHECK_QUESTION)
    print("Answer:  ", CHECK_ANSWER)
    print()
    show_tokens()
else:
    print("The request check did not pass. Fix TODO 1, then run both cells again.")

In [ ]:
# Pre-written. The email reader, and a small printer.
#
# parse_eml() turns one raw mail file into a dictionary of facts: the subject,
# the sender, the body text, the link hosts and more. Reading a mail file is
# not what this week teaches, so it arrives finished. You import it and call
# it. You never edit it.

from parse_eml import parse_eml


def email_path(email_id):
    """Turn an id such as "001" into the full path of that mail file."""
    matches = sorted(EMAIL_DIR.glob(email_id + "-*.eml"))
    if not matches:
        raise FileNotFoundError("No mail file starts with " + email_id
                                + " in " + str(EMAIL_DIR))
    return matches[0]


def show_email(parsed):
    """Print the subject, the sender, and the start of the body."""
    print("Subject:", parsed["subject"])
    print("From:   ", parsed["from_display"], "<" + str(parsed["from_address"]) + ">")
    print("Body, first 400 characters:")
    print(parsed["body_text"][:400])


# The instruction you put in front of an email body this week.
INSTRUCTION = ("Read the email below. In two sentences, say whether it looks "
               "like a phishing attempt, and why.")

email_001 = parse_eml(email_path("001"))
show_email(email_001)

In [ ]:
# SOLUTION. Not in the student notebook.

prompt_001 = INSTRUCTION + "\n\n" + email_001["body_text"]
answer_001 = ask_model(prompt_001)
print(answer_001)


In [ ]:
# Pre-written. One short prompt, six times at temperature 0.2 and six times
# at temperature 1.0. The per-call token line is switched off so that all
# twelve answers and both distinct-answer counts fit together on screen.
#
# This is the cell you screenshot.

# The request check in cell 8 already proved that TODO 1 forwards temperature.
# The pattern of answers is evidence about this run, not a code test.

PROMPT_001 = INSTRUCTION + "\n\n" + email_001["body_text"]
# Keep the prose prompt size from TODO 2 so the next cell can compare it.
PROMPT_TOKENS_001 = ((LAST_CALL["reply"] or {}).get("usage") or {}).get("prompt_tokens", 0)

VARIATION_INSTRUCTION = ("Read the email below. Answer with one word, "
                         "phishing or legitimate, then one short sentence saying why.")
VARIATION_PROMPT_001 = VARIATION_INSTRUCTION + "\n\n" + email_001["body_text"]

LOW_ANSWERS = []
HIGH_ANSWERS = []
SHOW_CALL_TOKENS = False
while len(LOW_ANSWERS) < 6:
    LOW_ANSWERS.append(ask_model(VARIATION_PROMPT_001, temperature=0.2))
while len(HIGH_ANSWERS) < 6:
    HIGH_ANSWERS.append(ask_model(VARIATION_PROMPT_001, temperature=1.0))
SHOW_CALL_TOKENS = True

LOW_DISTINCT = []
for answer in LOW_ANSWERS:
    if answer not in LOW_DISTINCT:
        LOW_DISTINCT.append(answer)
HIGH_DISTINCT = []
for answer in HIGH_ANSWERS:
    if answer not in HIGH_DISTINCT:
        HIGH_DISTINCT.append(answer)

print("--- temperature 0.2: six answers ---")
number = 0
while number < len(LOW_ANSWERS):
    print(f"{number + 1}. {LOW_ANSWERS[number]}")
    number = number + 1
print(f"Distinct answers at temperature 0.2: {len(LOW_DISTINCT)}")
print()
print("--- temperature 1.0: six answers ---")
number = 0
while number < len(HIGH_ANSWERS):
    print(f"{number + 1}. {HIGH_ANSWERS[number]}")
    number = number + 1
print(f"Distinct answers at temperature 1.0: {len(HIGH_DISTINCT)}")

In [ ]:
# Pre-written. A second email, sent the same way. Watch the token numbers.

start_run()

email_002 = parse_eml(email_path("002"))
show_email(email_002)
print()

PROMPT_002 = INSTRUCTION + "\n\n" + email_002["body_text"]
ANSWER_002 = ask_model(PROMPT_002)

print("--- Answer for email 002 ---")
print(ANSWER_002)
print()
print(f"Characters sent for email 001: {len(PROMPT_001)}")
print(f"Characters sent for email 002: {len(PROMPT_002)}")
print(f"Prompt tokens for email 001:   {PROMPT_TOKENS_001}")
print(f"Prompt tokens for email 002:   {RUN_USAGE['prompt']}")
print()
show_tokens()

## What to notice in what you just ran

Three things happened. Each one matters in a later week.

**The same question can give different answers.** You sent one short prompt
six times at each setting. The model picks each word from likely choices, so
one answer is one sample. It is not a fact, even when samples agree.

**Temperature can change how much answers differ.** A higher temperature
often allows more variation. Your small run may show more distinct answers,
the same number, or fewer at 1.0 than at 0.2. Every direction is a real
result, not a mistake. The request check in cell 8, not the answer pattern,
tells you whether your code passed the temperature setting.

**The token count moved with the length.** Email 002 has a longer body than
email 001. Its prompt cost more tokens. You pay for the text you send, not for
the number of questions you ask.

Your prose calls also show the hand-off to week 2: the model writes you a
sentence, and a sentence is hard for code to use.

Carry one thing into week 2. You cannot trust a single answer on its own.


In [ ]:
# Pre-written. This writes your submission file. Run it last.
#
# The block marked "your own first call" reads prompt_001 and answer_001 from
# TODO 2 if you made them. If you did not, the file says so rather than
# stopping.

LINES = []
LINES.append("ITSY 1377 week 1: first calls")
LINES.append("=============================")
LINES.append("")
LINES.append("Model configured: " + str(MODEL))
LINES.append("Emails used: 001, 002")
LINES.append("")
LINES.append("--- The prompt sent for email 001 ---")
LINES.append(PROMPT_001)
LINES.append("")
if not globals().get("prompt_001") or not globals().get("answer_001"):
    print("TODO 2 is not in this file. The export reads two names exactly:")
    print("   prompt_001   and   answer_001")
    print("If you used different names, rename yours in cell 10, run it, then")
    print("run this cell again. The file below is missing that section.")
    print()

LINES.append("--- Your own first call, from TODO 2 ---")
LINES.append("The prompt you built:")
LINES.append(str(globals().get("prompt_001", "not recorded")))
LINES.append("The answer it gave:")
LINES.append(str(globals().get("answer_001", "not recorded")))
LINES.append("")
LINES.append("--- Temperature prompt for email 001 ---")
LINES.append(VARIATION_PROMPT_001)
LINES.append("")
LINES.append("--- Six answers at temperature 0.2 ---")
for answer in LOW_ANSWERS:
    LINES.append(answer)
LINES.append(f"Distinct answers at temperature 0.2: {len(LOW_DISTINCT)}")
LINES.append("")
LINES.append("--- Six answers at temperature 1.0 ---")
for answer in HIGH_ANSWERS:
    LINES.append(answer)
LINES.append(f"Distinct answers at temperature 1.0: {len(HIGH_DISTINCT)}")
LINES.append("")
LINES.append("--- The prompt sent for email 002 ---")
LINES.append(PROMPT_002)
LINES.append("")
LINES.append("--- Answer for email 002 ---")
LINES.append(ANSWER_002)
LINES.append("")
LINES.append("--- Token totals for this session ---")
LINES.append(f"Calls counted:  {USAGE['calls']}")
LINES.append(f"Prompt tokens:  {USAGE['prompt']}")
LINES.append(f"Answer tokens:  {USAGE['answer']}")
LINES.append(f"Total tokens:   {USAGE['total']}")
LINES.append(f"Prompt tokens, email 001: {PROMPT_TOKENS_001}")
LINES.append(f"Prompt tokens, email 002: {RUN_USAGE['prompt']}")

save_artifact("week1_first_calls.txt", "\n".join(LINES))

# What is left of your allowance, after this week's work.
budget_left()

# Your work lives only in this browser. Bundle it up so you have a copy.
backup_my_work()


## What to submit

Three items. Nothing else. No folders and no links.

1. **A screenshot** of the output of the temperature cell. It must show both
   labelled groups, all twelve answers, and both distinct-answer counts. That
   cell prints no key material.
2. **The file** `week1_first_calls.txt`. The export cell printed its full path.
   Attach that file.
3. **A written answer** of three to five sentences, typed into the submission
   box.

Answer all three questions in the written answer.

1. How many distinct answers appeared at each temperature? Did the 1.0 group
   show more, the same number, or fewer than the 0.2 group?
2. What happened to the token count on email 002, and why?
3. What would you check before trusting this answer in a real ticket?

Your discussion post uses the same file.

**If the model never answered.** The notebook still runs and still writes the
file. Run the configuration cell again and read what it prints. Then email your
instructor and say what it said.
